In [1]:
import sqlite3

In [3]:
import sqlite3
from datetime import date

def init_database():
    conn = sqlite3.connect('company.db')
    cursor = conn.cursor()
    
    # 创建部门表
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Dept (
        dno INTEGER PRIMARY KEY,
        dname TEXT NOT NULL,
        budget REAL CHECK(budget >= 0),
        manager INTEGER,
        FOREIGN KEY (manager) REFERENCES Emp(eno)
    )
    ''')
    
    # 创建员工表
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Emp (
        eno INTEGER PRIMARY KEY,
        ename TEXT NOT NULL,
        birthday DATE,
        level INTEGER CHECK(level BETWEEN 1 AND 5),
        position TEXT,
        salary REAL CHECK(salary >= 0),
        dno INTEGER,
        FOREIGN KEY (dno) REFERENCES Dept(dno),
        CHECK (
            (level = 1 AND salary BETWEEN 0 AND 5000) OR
            (level = 2 AND salary BETWEEN 5001 AND 10000) OR
            (level = 3 AND salary BETWEEN 10001 AND 15000) OR
            (level = 4 AND salary BETWEEN 15001 AND 20000) OR
            (level = 5 AND salary > 20000)
        )
    )
    ''')
    
    # 创建编码对照表
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS CodeMapping (
        code_type TEXT NOT NULL,
        code_value INTEGER NOT NULL,
        description TEXT NOT NULL,
        PRIMARY KEY (code_type, code_value)
    )
    ''')
    
    # 插入编码对照数据
    code_mappings = [
        ('level', 1, '初级'),
        ('level', 2, '中级'),
        ('level', 3, '高级'),
        ('level', 4, '专家'),
        ('level', 5, '资深专家'),
        ('position', 1, '工程师'),
        ('position', 2, '高级工程师'),
        ('position', 3, '经理'),
        ('position', 4, '总监'),
        ('position', 5, '副总裁'),
        ('position', 6, '总裁'),
        ('dept', 1, '研发部'),
        ('dept', 2, '市场部'),
        ('dept', 3, '财务部'),
        ('dept', 4, '人力资源部'),
        ('dept', 5, '行政部')
    ]
    
    cursor.executemany('INSERT OR IGNORE INTO CodeMapping VALUES (?, ?, ?)', code_mappings)
    
    conn.commit()
    return conn


In [4]:
def insert_mutually_referencing_rows(conn):
    cursor = conn.cursor()
    
    try:
        # 禁用外键检查
        cursor.execute("PRAGMA foreign_keys = OFF")
        
        # 先插入部门（manager暂时为NULL）
        cursor.execute("INSERT INTO Dept (dno, dname, budget, manager) VALUES (1, '研发部', 1000000, NULL)")
        
        # 插入员工（引用该部门）
        cursor.execute("INSERT INTO Emp (eno, ename, birthday, level, position, salary, dno) VALUES (101, '张三', '1990-05-15', 3, '高级工程师', 12000, 1)")
        
        # 更新部门的manager字段
        cursor.execute("UPDATE Dept SET manager = 101 WHERE dno = 1")
        
        conn.commit()
        print("成功插入互相引用的行")
        
    except sqlite3.Error as e:
        conn.rollback()
        print("插入失败:", e)
    finally:
        # 重新启用外键检查
        cursor.execute("PRAGMA foreign_keys = ON")

In [5]:
def test_salary_level_constraint(conn):
    cursor = conn.cursor()
    
    # 正测试 - 符合约束的插入
    try:
        cursor.execute("INSERT INTO Emp (eno, ename, level, salary) VALUES (102, '李四', 2, 8000)")
        print("正测试: 成功插入有效数据 (level=2, salary=8000)")
    except sqlite3.Error as e:
        print("正测试失败:", e)
    
    # 负测试 - 违反约束的插入
    try:
        cursor.execute("INSERT INTO Emp (eno, ename, level, salary) VALUES (103, '王五', 1, 6000)")
        print("负测试失败: 不应该允许 level=1 但 salary=6000 的插入")
    except sqlite3.IntegrityError as e:
        print("负测试: 成功捕获违反约束的操作 -", e)
    
    conn.rollback()  # 回滚测试数据

In [6]:
def generate_employee_code(conn, eno):
    cursor = conn.cursor()
    
    # 获取员工信息
    cursor.execute('''
    SELECT e.eno, e.ename, e.birthday, e.level, e.position, e.salary, e.dno, d.dname
    FROM Emp e LEFT JOIN Dept d ON e.dno = d.dno
    WHERE e.eno = ?
    ''', (eno,))
    
    employee = cursor.fetchone()
    if not employee:
        return None
    
    # 获取编码对照
    code_parts = []
    
    # 1-4位: 员工编号 (4位，不足补零)
    code_parts.append(f"{employee[0]:04d}")
    
    # 5-8位: 部门编码 (从CodeMapping获取)
    cursor.execute("SELECT code_value FROM CodeMapping WHERE code_type = 'dept' AND description = ?", (employee[7],))
    dept_code = cursor.fetchone()
    code_parts.append(f"{dept_code[0]:02d}" if dept_code else "00")
    
    # 9-12位: 出生年份
    birthday = date.fromisoformat(employee[2]) if employee[2] else date.today()
    code_parts.append(f"{birthday.year:04d}")
    
    # 13-14位: 职位编码
    cursor.execute("SELECT code_value FROM CodeMapping WHERE code_type = 'position' AND description = ?", (employee[4],))
    position_code = cursor.fetchone()
    code_parts.append(f"{position_code[0]:02d}" if position_code else "00")
    
    # 15-16位: 级别编码
    code_parts.append(f"{employee[3]:02d}")
    
    # 17-20位: 工资等级 (工资/1000的整数部分)
    salary_grade = int(employee[5] / 1000) if employee[5] else 0
    code_parts.append(f"{salary_grade:04d}")
    
    # 组合所有部分
    employee_code = ''.join(code_parts)
    return employee_code

In [7]:
def test():
    conn = init_database()
    
    print("\n=== 测试互相引用约束 ===")
    insert_mutually_referencing_rows(conn)
    
    print("\n=== 测试工资级别约束 ===")
    test_salary_level_constraint(conn)
    
    # 插入一些测试数据
    cursor = conn.cursor()
    cursor.execute("INSERT OR IGNORE INTO Dept (dno, dname, budget) VALUES (2, '市场部', 800000)")
    cursor.execute("INSERT OR IGNORE INTO Emp (eno, ename, birthday, level, position, salary, dno) VALUES (201, '赵六', '1985-11-20', 4, '总监', 18000, 2)")
    conn.commit()
    
    print("\n=== 测试智能码生成 ===")
    employee_code = generate_employee_code(conn, 201)
    print(f"生成的员工智能码: {employee_code}")
    
    # 解释智能码
    if employee_code:
        print("\n智能码解析:")
        print(f"员工编号: {employee_code[0:4]}")
        print(f"部门编码: {employee_code[4:6]} (市场部)")
        print(f"出生年份: {employee_code[6:10]}")
        print(f"职位编码: {employee_code[10:12]} (总监)")
        print(f"级别编码: {employee_code[12:14]} (4)")
        print(f"工资等级: {employee_code[14:18]} (18)")
    
    conn.close()


In [8]:
test()


=== 测试互相引用约束 ===
成功插入互相引用的行

=== 测试工资级别约束 ===
正测试: 成功插入有效数据 (level=2, salary=8000)
负测试: 成功捕获违反约束的操作 - CHECK constraint failed: (level = 1 AND salary BETWEEN 0 AND 5000) OR
            (level = 2 AND salary BETWEEN 5001 AND 10000) OR
            (level = 3 AND salary BETWEEN 10001 AND 15000) OR
            (level = 4 AND salary BETWEEN 15001 AND 20000) OR
            (level = 5 AND salary > 20000)

=== 测试智能码生成 ===
生成的员工智能码: 020102198504040018

智能码解析:
员工编号: 0201
部门编码: 02 (市场部)
出生年份: 1985
职位编码: 04 (总监)
级别编码: 04 (4)
工资等级: 0018 (18)
